In [1]:
from Bio import AlignIO
from Bio import Align
import numpy as np
from src import alignment

In [2]:
alnmt: Align.MultipleSeqAlignment = AlignIO.read('data/cox2_th/alignment/msa.fasta', 'fasta')
#alnmt: Align.MultipleSeqAlignment = AlignIO.read('gappiness-test.fasta', 'fasta')

In [3]:
def msa_to_numpy(msa: Align.MultipleSeqAlignment):
    seq_strings = [str(record.seq) for record in msa]
    return np.array([list(seq) for seq in seq_strings])
ali_mat = msa_to_numpy(alnmt)
num_seqs, aln_len = ali_mat.shape
ali_mat

array([['T', 'G', 'G', ..., 'A', 'A', 'T'],
       ['T', 'T', 'G', ..., 'A', 'A', 'T'],
       ['T', 'T', 'G', ..., 'A', 'A', 'T'],
       ...,
       ['-', '-', '-', ..., 'A', 'A', 'T'],
       ['-', '-', '-', ..., 'A', 'A', 'T'],
       ['-', '-', '-', ..., 'A', 'A', 'T']],
      shape=(1322, 319), dtype='<U1')

In [4]:
residue_mask = (ali_mat != '-')
residues_per_column: np.ndarray = np.sum(residue_mask, axis=0)
residues_per_column

array([  13,   13,   12,    6,    1,    1,    1,    1,    4,   22,   23,
         44,   60,  104,  143,  173,  191,  227,  244,  249,  256,  261,
        296,  311,  317,  692,  741,  743,  759,  761,  785,   54,  749,
        790,  797,  791,  810,  814,  816, 1077, 1080, 1083, 1086, 1087,
       1088, 1118, 1118, 1121, 1133, 1138, 1143, 1165, 1170, 1170, 1176,
       1177, 1185, 1216, 1217, 1222, 1239, 1239, 1243, 1246, 1253, 1256,
       1261, 1262, 1262, 1264, 1263, 1265, 1267, 1270, 1275, 1277, 1281,
       1281, 1283, 1286, 1287, 1287, 1288, 1288, 1289, 1289, 1289,    7,
          7,    7, 1291, 1291, 1296, 1298, 1299, 1300, 1301, 1302, 1303,
       1304, 1305, 1306, 1310, 1310, 1310, 1314, 1314, 1314, 1318, 1318,
       1318, 1318, 1319, 1319, 1320, 1320, 1320, 1321, 1321, 1322, 1322,
       1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322,
       1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322, 1322,
       1322, 1322, 1322, 1322, 1322, 1322, 1312, 13

In [5]:
# only consider columns with more gaps than residues
more_gaps_than_residues_mask = residues_per_column < (num_seqs / 2)
residues_per_column = np.where(more_gaps_than_residues_mask, residues_per_column, 0)
residues_per_column

array([ 13,  13,  12,   6,   1,   1,   1,   1,   4,  22,  23,  44,  60,
       104, 143, 173, 191, 227, 244, 249, 256, 261, 296, 311, 317,   0,
         0,   0,   0,   0,   0,  54,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   7,   7,   7,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [6]:
f_mat = np.zeros((num_seqs, num_seqs), dtype=np.int32)



In [7]:
residues_per_column_mat = np.broadcast_to(residues_per_column, (num_seqs, aln_len))

In [8]:
# mask gaps with 0
residues_per_column_mat_filtered = np.where(residue_mask, residues_per_column_mat, 0)
residues_per_column_mat_filtered

array([[13, 13, 12, ...,  0,  0,  0],
       [13, 13, 12, ...,  0,  0,  0],
       [13, 13, 12, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0]], shape=(1322, 319))

In [9]:
# source: https://stackoverflow.com/questions/34755244/apply-bincount-to-each-row-of-a-2d-numpy-array
N: int = num_seqs + 1
id = residues_per_column_mat_filtered + (N * np.arange(residues_per_column_mat_filtered.shape[0]))[:, None]
residue_hist = np.bincount(id.ravel(), minlength=N * residues_per_column_mat_filtered.shape[0]).reshape(-1, N)[:,1:]
id

array([[     13,      13,      12, ...,       0,       0,       0],
       [   1336,    1336,    1335, ...,    1323,    1323,    1323],
       [   2659,    2659,    2658, ...,    2646,    2646,    2646],
       ...,
       [1745037, 1745037, 1745037, ..., 1745037, 1745037, 1745037],
       [1746360, 1746360, 1746360, ..., 1746360, 1746360, 1746360],
       [1747683, 1747683, 1747683, ..., 1747683, 1747683, 1747683]],
      shape=(1322, 319))

In [10]:
n_vals = np.arange(1, num_seqs + 1)
weights = (num_seqs - n_vals)
weights

array([1321, 1320, 1319, ...,    2,    1,    0], shape=(1322,))

In [11]:
numerator = (weights * residue_hist).sum(axis=1)
numerator

array([30077, 20876, 20876, ...,     0,     0,     0], shape=(1322,))

In [12]:
gappiness:np.ndarray = (residue_hist * weights).sum(axis=1) / (aln_len * num_seqs)
gappiness

array([0.07132017, 0.04950227, 0.04950227, ..., 0.        , 0.        ,
       0.        ], shape=(1322,))

In [13]:
%%time
ali_mat = msa_to_numpy(alnmt)
num_seqs, aln_len = ali_mat.shape
residue_mask = (ali_mat != '-')
residues_per_column: np.ndarray = np.sum(residue_mask, axis=0)
# only consider columns with more gaps than residues
more_gaps_than_residues_mask = residues_per_column < (num_seqs / 2)
residues_per_column = np.where(more_gaps_than_residues_mask, residues_per_column, 0)
f_mat = np.zeros((num_seqs, num_seqs), dtype=np.int32)
residues_per_column_mat = np.broadcast_to(residues_per_column, (num_seqs, aln_len))
residues_per_column_mat_filtered = np.where(residue_mask, residues_per_column_mat, 0)
N: int = num_seqs + 1
id = residues_per_column_mat_filtered + (N * np.arange(residues_per_column_mat_filtered.shape[0]))[:, None]
residue_hist = np.bincount(id.ravel(), minlength=N * residues_per_column_mat_filtered.shape[0]).reshape(-1, N)[:,1:]
n_vals = np.arange(1, num_seqs + 1)
weights = (num_seqs - n_vals)
gappiness: np.ndarray = (residue_hist * weights).sum(axis=1) / (aln_len * num_seqs)
gappiness

CPU times: user 40 ms, sys: 5.18 ms, total: 45.2 ms
Wall time: 44.8 ms


array([0.07132017, 0.04950227, 0.04950227, ..., 0.        , 0.        ,
       0.        ], shape=(1322,))

In [14]:
%%time
alignment._compute_gappiness(alnmt)

CPU times: user 46.9 ms, sys: 4.81 ms, total: 51.7 ms
Wall time: 52.5 ms


array([0.07132017, 0.04950227, 0.04950227, ..., 0.        , 0.        ,
       0.        ], shape=(1322,))